## K-NN CLASSIFIER

k-NN finds the k most historical matches and uses them to decide who is likely to win

In [ ]:
from pathlib import Path 

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

from sklearn.preprocessing import StandardScaler # because k-nn works with distances features need to have comparable scales
from sklearn.neighbors import KNeighborsClassifier

import sys
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    
from src.preprocessing import (
    fit_preprocessor,
    transform_data,
    categorical_features,
    numeric_features
)

from src.temporal_cv import (
    create_temporal_folds,
    validation_years
)

from src.evaluation import (
    evaluate_model
)

from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay
)

In [ ]:
PROCESSED_DATA_DIR = Path("../data/processed")

input_file = (
    PROCESSED_DATA_DIR / "feature_split_matches.csv"
)

matches = pd.read_csv(
    input_file, parse_dates = ["Date"]
)

matches = matches.sort_values(
    by=["Date", "MatchID"]
).reset_index(drop=True)

print("Dataset shape:", matches.shape)
print(matches["DataSplit"].value_counts())

In [ ]:
## create copies of the sets 
train_data = matches.loc[
    matches["DataSplit"] == "Training"
].copy()

validation_data = matches.loc[
    matches["DataSplit"] == "Validation"
].copy()

test_data = matches.loc[
    matches["DataSplit"] == "Test"
].copy()

## check the periods
print(
    "Training:", 
    train_data["Date"].min(),
    "-",
    train_data["Date"].max()
)

print(
    "Validating:", 
    validation_data["Date"].min(),
    "-",
    validation_data["Date"].max()
)

print(
    "Testing:", 
    test_data["Date"].min(),
    "-",
    test_data["Date"].max()
)


In [ ]:
assert(
    train_data["Date"].max() < validation_data["Date"].min()
)

assert(
    validation_data["Date"].max() < test_data["Date"].min()
)

print("Chronological order is maintained.")

In [ ]:
# The feature lists are imported from preprocessing.py.
# This ensures that all prediction models use
# exactly the same starting feature set.

print(
    "Numerical features:",
    len(numeric_features)
)

print(
    "Categorical features:",
    len(categorical_features)
)

print(
    "Raw features:",
    len(numeric_features) + len(categorical_features)
)


In [ ]:
##create temporal cross-validation folds for the training set
temporal_folds= create_temporal_folds(
    train_data,
    validation_years = validation_years
)

print(
    "Number of temporal folds created:", 
    len(temporal_folds)
)

pd.DataFrame(temporal_folds)

In [ ]:
##decide which values of k we want to try

## weights = "uniform" means every neighbor has the same vote
## weights = "distance" means closer neighbors have more influence

configurations = [
    {
        "name": "kNN k=11",
        "n_neighbors": 11,
        "weights": "uniform"
    },

    {
        "name": "kNN k=31",
        "n_neighbors": 31,
        "weights": "uniform"
    },

    {
        "name": "kNN k=51",
        "n_neighbors": 51,
        "weights": "uniform"
    },

    {
        "name": "kNN k=75",
        "n_neighbors": 75,
        "weights": "uniform"
    },

    {
        "name": "kNN k=101",
        "n_neighbors": 101,
        "weights": "uniform"
    },

    {
        "name": "kNN k=151",
        "n_neighbors": 151,
        "weights": "uniform"
    },

    {
        "name": "kNN k=11 distance",
        "n_neighbors": 11,
        "weights": "distance"
    },

    {
        "name": "kNN k=31 distance",
        "n_neighbors": 31,
        "weights": "distance"
    },

    {
        "name": "kNN k=51 distance",
        "n_neighbors": 51,
        "weights": "distance"
    },

    {
        "name": "kNN k=75 distance",
        "n_neighbors": 75,
        "weights": "distance"
    },

    {
        "name": "kNN k=101 distance",
        "n_neighbors": 101,
        "weights": "distance"
    },

    {
        "name": "kNN k=151",
        "n_neighbors": 151,
        "weights": "distance"
    }
]

In [ ]:
print(
    "Configurations to test:",
    len(configurations)
)

now we create a list where we save  the results of each configuration and each fold

we have 12 configurations x 4 folds = 48 experiments in total

In [ ]:
knn_cv_results = [] 

for configuration in configurations:
    print("Testing:",configuration["name"])

    for fold in temporal_folds:
        fold_number = (fold["Fold"])
        validation_year = (fold["ValidationYear"])

        fold_train = train_data.loc[
            train_data["Date"].dt.year < validation_year
        ].copy()

        fold_validation = train_data.loc[
            train_data["Date"].dt.year == validation_year
        ].copy()


        (
            fold_numeric_medians,
            fold_categorical_modes,
            fold_encoder
        ) = fit_preprocessor (
            fold_train
        )

        X_fold_train = transform_data(
            fold_train,
            fold_numeric_medians,
            fold_categorical_modes,
            fold_encoder
        )

        X_fold_validation = transform_data(
            fold_validation,
            fold_numeric_medians,
            fold_categorical_modes,
            fold_encoder
        )

        # remember Player1Won = 1 means Player 1 won, Player1Won = 0 means Player 2 won
        y_fold_train = (fold_train["Player1Won"].to_numpy())
        y_fold_validation = (fold_validation["Player1Won"].to_numpy())

        fold_scaler = StandardScaler()

        X_fold_train[numeric_features] = fold_scaler.fit_transform(
            X_fold_train[numeric_features]
        )

        X_fold_validation[numeric_features] = fold_scaler.transform(
            X_fold_validation[numeric_features]
        )

        # we only use transform() for the validation because the scaler learns the means and std from the training set
        # we never do fit_tranform(X_fold_validation) because that would use validation information which would introduce bias

        current_model = KNeighborsClassifier(
            n_neighbors = configuration["n_neighbors"],
            weights = configuration["weights"],
            n_jobs = -1
        )

        current_model.fit(
            X_fold_train,
            y_fold_train
        )

        current_predictions = (
            current_model.predict(X_fold_validation)
        )

        current_probabilities = (
            current_model.predict_proba(X_fold_validation)[:, 1]
        )

        # the [:, 1] gives the probbaility of class 1

        current_fold_results = (
            evaluate_model(
                y_true = y_fold_validation,
                y_pred = current_predictions,
                model_name = configuration["name"],
                y_probability = current_probabilities
            ) # calculates accuracy, balanced accuracy, precision, etc.
        )

        current_fold_result = (
            current_fold_results.iloc[0].to_dict()
        )

        current_fold_result["Fold"] = fold_number
        current_fold_result["ValidationYear"] = validation_year
        current_fold_result["Neighbors"] = configuration["n_neighbors"]
        current_fold_result["Weights"] = configuration["weights"]

        knn_cv_results.append(current_fold_result)

        print(
            "Fold", 
            fold_number,
            "year",
            validation_year,
            "accuracy:",
            round(current_fold_result["Accuracy"], 4)
        )

print(
    "\nTotal results collected:",
    len(knn_cv_results)
)

**ANALYSIS**

we notice that with an increasing k our accuracy increases until we reach a value of k = 201, where with a uniform weight across all values we notice a decrease, while with values weighted differently it maintains a uniform andamento

In [ ]:
# convert results into a dataframe

knn_cv_fold_results = pd.DataFrame(knn_cv_results)

print("Configurations: ", len(configurations))
print("Temporal folds: ", len(temporal_folds))
print("Results: ", len(knn_cv_fold_results))

In [ ]:
assert(len(knn_cv_fold_results) == len(configurations) * len(temporal_folds))

In [ ]:
knn_cv_fold_results.head()

In [ ]:
print(len(knn_cv_fold_results))

In [ ]:
knn_cv_summary = (
    knn_cv_fold_results.groupby(
        ["Model", "Neighbors", "Weights"],
        as_index = False
    ).agg(
        MeanAccuracy = ("Accuracy", "mean"),
        StdAccuracy = ("Accuracy", "std"),
        MeanBalancedAccuracy = ("BalancedAccuracy", "mean"),
        MeanF1Score = ("F1", "mean"),
        MeanROC_AUC = ("ROC-AUC", "mean"),
        MeanLogLoss = ("LogLoss", "mean")
    )
)

In [ ]:
knn_cv_summary = (
    knn_cv_summary.sort_values(
        by = ["MeanAccuracy", "MeanROC_AUC"],
        ascending = [False, False]
    ).reset_index(drop = True)
)

In [ ]:
knn_cv_summary

In [ ]:
# select the best configuration automatically

best_model_name = (knn_cv_summary.iloc[0]["Model"])

best_configuration = next(
    configuration
    for configuration in configurations
    if configuration["name"] == best_model_name
)

print("Selected k-NN: best_model_name")

best_configuration

we are now done with hyperparameter selection so now we can use the complete 2015-2022 training set to train the selected model

In [ ]:
(
    numeric_medians,
    categorical_modes,
    one_hot_encoder
) = fit_preprocessor(
    train_data
)

X_train = transform_data(
    train_data,
    numeric_medians,
    categorical_modes,
    one_hot_encoder
)

X_validation = transform_data(
    validation_data,
    numeric_medians,
    categorical_modes,
    one_hot_encoder
)

y_train = (train_data["Player1Won"].to_numpy())
y_validation = (validation_data["Player1Won"].to_numpy())

In [ ]:
print(
    "Training X:",
    X_train.shape
)

print(
    "Validation X:",
    X_validation.shape
)

In [ ]:
assert (X_train.isna().sum().sum() == 0)
assert (X_validation.isna().sum().sum() == 0)

In [ ]:
knn_scaler = StandardScaler()

X_train[numeric_features] = knn_scaler.fit_transform(
    X_train[numeric_features]
)

X_validation[numeric_features] = knn_scaler.transform(
    X_validation[numeric_features]
)

print("k-NN scaling completed")

In [ ]:
best_knn = KNeighborsClassifier( # build the best knn
    n_neighbors = best_configuration["n_neighbors"],
    weights = best_configuration["weights"],
    n_jobs = -1
)

best_knn.fit(X_train, y_train)

In [ ]:
# classification report

best_validation_predictions = (
    best_knn.predict(X_validation)
) # returns predicted classes [0, 1, 1, 0, 1, ...]

best_validation_probabilities = (
    best_knn.predict_proba(X_validation)[:, 1]
)# returns probabilities that player 1 wins [0.23, 0.81, ...]

print(
    classification_report(
        y_validation,
        best_validation_predictions,
        target_names = ["Player 2 wins", "Player 1 wins"],
        zero_division = 0
    )# compares true validation labels against predicted validation labels
)

In [ ]:
best_knn_result = evaluate_model(
    y_true = y_validation,
    y_pred = (best_validation_predictions),
    model_name = (best_model_name),
    y_probability = (best_validation_probabilities)
)

best_knn_result

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_validation,
    best_validation_predictions,
    display_labels = ["Player 2 wins", "Player 1 wins"],
    values_format = "d"
)

plt.title("Selected k-NN validation confusion matrix")
plt.show()

In [ ]:
for weight_type in["uniform", "distance"]:
    plot_data = (
        knn_cv_summary.loc[
            knn_cv_summary["Weights"] == weight_type
        ].sort_values("Neighbors")
    )

    plt.plot(
        plot_data["Neighbors"],
        plot_data["MeanAccuracy"],
        marker = "o",
        label = weight_type
    )

plt.xlabel("Number of neighbors (k)")
plt.ylabel("Mean temporal-CV accuracy")
plt.title("k-NN tuning")
plt.legend()
plt.show()

In [ ]:
MODEL_SELECTION_RESULTS_DIR = Path(
    "../results/model_selection"
)

VALIDATION_RESULTS_DIR = Path(
    "../results/validation"
)

MODELS_DIR = Path("../models")

knn_cv_fold_results.to_csv(
    MODEL_SELECTION_RESULTS_DIR 
    / "knn_cv_fold_results.csv",
    index = False
)

knn_cv_summary.to_csv(
    MODEL_SELECTION_RESULTS_DIR
    / "knn_cv_tuning_summary.csv",
    index = False
)

best_knn_result.to_csv(
    VALIDATION_RESULTS_DIR
    / "best_knn_result.csv",
    index=False
)

In [ ]:
import joblib

joblib.dump(
    best_knn,
    MODELS_DIR / "best_knn_validation.joblib"
)

joblib.dump(
    knn_scaler,
    MODELS_DIR / "best_knn_scaler.joblib"
)

In [ ]:
print("k-NN TEMPORAL CV")

print(knn_cv_summary)

print("\nSelected configuration:", best_model_name)

print("\nExternal 2023 validation")

print(best_knn_result)